# Cambiar un usuario

## Qué hace

Cambia el nombre de un usuario ya guardado. Si no se manda email, se queda el que había.

## Qué pasa en la base de datos

Actualiza esa fila y la fecha de cambio. El id no cambia.

## Qué acción usa

`PATCH /users/:id` solo con el nombre Ada Lovelace.

## Anotaciones

Si no está, se ve `User not found`. Si el nombre está vacío, se ve `Name is empty`. Si el email no vale, se ve `Email is invalid`. Si el email es de otro usuario, se ve `Email is already used`. Si no se llega a la base, se ve `Could not update the user`. Se abre con `pnpm jupyter-node-http` desde la app.


In [1]:
import os
import subprocess

code = r"""
import Table from "cli-table3";

const origin = "http://127.0.0.1:3004";
const usersResponse = await fetch(origin + "/users");
const users = await usersResponse.json();
let adaId;
for (const user of users) {
  if (user.email === "ada@example.com") {
    adaId = user.id;
  }
}
const response = await fetch(origin + "/users/" + adaId, {
  method: "PATCH",
  headers: {
    "content-type": "application/json",
  },
  body: JSON.stringify({
    name: "Ada Lovelace",
  }),
});
const result = await response.json();
if (response.ok === false) {
  console.log(result.message);
} else {
  const table = new Table({
    head: ["id", "name", "email", "createdAt", "updatedAt"],
    style: { head: ["cyan", "bold"] },
  });
  table.push([
    result.id,
    result.name,
    result.email,
    result.createdAt,
    result.updatedAt,
  ]);
  console.log(table.toString());
}
"""

env = os.environ.copy()
env["FORCE_COLOR"] = "1"
env.pop("NO_COLOR", None)

completed = subprocess.run(
    ["node", "--input-type=module", "-e", code],
    capture_output=True,
    text=True,
    env=env,
)
print(completed.stdout, end="")
if completed.returncode != 0:
    print(completed.stderr, end="")


┌────┬──────────────┬─────────────────┬─────────────────────┬─────────────────────┐
│ id │ name         │ email           │ createdAt           │ updatedAt           │
├────┼──────────────┼─────────────────┼─────────────────────┼─────────────────────┤
│ 25 │ Ada Lovelace │ ada@example.com │ 2026-11-05 13:07:39 │ 2026-11-05 13:07:41 │
└────┴──────────────┴─────────────────┴─────────────────────┴─────────────────────┘
